# Buscador Semántico de Imágenes por Texto
## Pipeline Completo: OpenCLIP + FAISS

Este notebook documenta el proceso completo de construcción del sistema de búsqueda:
desde la descarga del dataset hasta la evaluación cuantitativa con Precision@K y Recall@K.

**Pipeline:**
```
1. Dataset (COCO val2017)
2. OpenCLIP ViT-B-32 (embeddings L2-normalizados)
3. FAISS IndexFlatIP (búsqueda exacta por cosine similarity)
4. Consulta textual (Top-K imágenes relevantes)
5. Evaluación (Precision@K / Recall@K)
```

| Componente | Decisión | Justificación |
|---|---|---|
| Modelo | OpenCLIP ViT-B-32 / laion2b_s34b_b79k | Balance velocidad/calidad |
| Índice | FAISS IndexFlatIP | Búsqueda exacta, sin parámetros, reproducible |
| Dataset | COCO 2017 val (5.000 imgs) | 80 categorías oficiales como ground truth |

---
## 1. Setup

Ejecutar la celda de instalación de dependencias.

In [ ]:
!pip install -q open_clip_torch faiss-cpu torch torchvision numpy pandas Pillow tqdm requests

print("Entorno listo.")

In [ ]:
import json
import pickle
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import torch
import open_clip
import faiss
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

print(f"PyTorch : {torch.__version__}")
print(f"FAISS   : {faiss.__version__}")

In [ ]:
# Configuración
class _Cfg:
    DATA_DIR              = Path("data")
    IMAGES_DIR            = Path("data/images")
    METADATA_PATH         = Path("data/metadata.csv")
    ANNOTATIONS_PATH      = Path("data/annotations/instances_val2017.json")
    EMBEDDINGS_DIR        = Path("embeddings")
    IMAGE_EMBEDDINGS_PATH = Path("embeddings/image_embeddings.npy")
    IMAGE_PATHS_PATH      = Path("embeddings/image_paths.pkl")
    FAISS_INDEX_PATH      = Path("embeddings/faiss_index.bin")
    MODEL_NAME            = "ViT-B-32"
    PRETRAINED            = "laion2b_s34b_b79k"
    EMBEDDING_DIM         = 512
    BATCH_SIZE            = 32
    TOP_K_DEFAULT         = 5
    DEVICE                = "cuda" if torch.cuda.is_available() else "cpu"

config = _Cfg()
print(f"Device  : {config.DEVICE}")
print(f"Model   : {config.MODEL_NAME} / {config.PRETRAINED}")

# Funciones del pipeline

def load_model():
    model, _, preprocess = open_clip.create_model_and_transforms(
        config.MODEL_NAME, pretrained=config.PRETRAINED
    )
    tokenizer = open_clip.get_tokenizer(config.MODEL_NAME)
    model = model.to(config.DEVICE).eval()
    return model, preprocess, tokenizer

class _ImageDataset(Dataset):
    def __init__(self, image_paths, preprocess):
        self.image_paths = image_paths
        self.preprocess = preprocess
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        return self.preprocess(Image.open(self.image_paths[idx]).convert("RGB"))

def encode_images(image_paths, model, preprocess):
    loader = DataLoader(_ImageDataset(image_paths, preprocess), batch_size=config.BATCH_SIZE, num_workers=0)
    parts = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Codificando imagenes"):
            f = model.encode_image(batch.to(config.DEVICE))
            parts.append((f / f.norm(dim=-1, keepdim=True)).cpu().numpy())
    return np.vstack(parts).astype(np.float32)

def encode_text(query, model, tokenizer):
    tokens = tokenizer([query]).to(config.DEVICE)
    with torch.no_grad():
        f = model.encode_text(tokens)
        f = f / f.norm(dim=-1, keepdim=True)
    return f.cpu().numpy().astype(np.float32)

def parse_coco_annotations(path):
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)

def build_image_category_index(annotations):
    idx = {}
    for ann in annotations["annotations"]:
        idx.setdefault(ann["image_id"], set()).add(ann["category_id"])
    return idx

def build_index(embeddings):
    index = faiss.IndexFlatIP(config.EMBEDDING_DIM)
    index.add(embeddings)
    return index

def save_index(index, path):
    faiss.write_index(index, str(path))

def load_index(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Índice FAISS no encontrado en '{path}'.")
    return faiss.read_index(str(path))

def _is_relevant(image_path, target_category_id, category_index):
    return target_category_id in category_index.get(int(Path(image_path).stem), set())

def count_relevant_images(target_category_id, category_index):
    return sum(1 for cats in category_index.values() if target_category_id in cats)

def precision_at_k(results, target_category_id, k, category_index):
    top_k = results[:k]
    if not top_k:
        return 0.0
    return sum(1 for r in top_k if _is_relevant(r["image_path"], target_category_id, category_index)) / k

def recall_at_k(results, target_category_id, k, category_index, total_relevant):
    if total_relevant == 0:
        return 0.0
    top_k = results[:k]
    return sum(1 for r in top_k if _is_relevant(r["image_path"], target_category_id, category_index)) / total_relevant

def evaluate(search_engine, eval_queries, category_index, k_values=None):
    if k_values is None:
        k_values = [1, 3, 5]
    max_k = max(k_values)
    query_results = []
    for q in tqdm(eval_queries, desc="Evaluando"):
        results = search_engine.search(q["query_text"], k=max_k)
        target_id = q["target_category_id"]
        total_relevant = count_relevant_images(target_id, category_index)
        metrics = {}
        for k in k_values:
            metrics[f"precision_at_{k}"] = precision_at_k(results, target_id, k, category_index)
            metrics[f"recall_at_{k}"] = recall_at_k(results, target_id, k, category_index, total_relevant)
        query_results.append({
            "query_text": q["query_text"],
            "target_category": q["target_category"],
            "target_category_id": target_id,
            "total_relevant_in_dataset": total_relevant,
            "results": results,
            "relevance_flags": [_is_relevant(r["image_path"], target_id, category_index) for r in results],
            **metrics,
        })
    aggregates = {}
    for k in k_values:
        aggregates[f"avg_precision_at_{k}"] = sum(r[f"precision_at_{k}"] for r in query_results) / len(query_results)
        aggregates[f"avg_recall_at_{k}"]    = sum(r[f"recall_at_{k}"]    for r in query_results) / len(query_results)
    success = aggregates.get("avg_precision_at_5", 0.0) >= 0.70
    return {"query_results": query_results, **aggregates, "success": success, "success_threshold": 0.70, "total_queries": len(query_results)}

class SearchEngine:
    def __init__(self, index_path=None, image_paths_path=None):
        self.index = load_index(index_path or config.FAISS_INDEX_PATH)
        ip = Path(image_paths_path or config.IMAGE_PATHS_PATH)
        if not ip.exists():
            raise FileNotFoundError(f"Rutas de imágenes no encontradas en '{ip}'.")
        with open(ip, "rb") as fh:
            self.image_paths = pickle.load(fh)
        self.model, _, self.tokenizer = load_model()

    def search(self, query, k=None):
        if k is None:
            k = config.TOP_K_DEFAULT
        if not query or not query.strip():
            raise ValueError("La consulta no puede estar vacía.")
        query_emb = encode_text(query.strip(), self.model, self.tokenizer)
        actual_k = min(k, self.index.ntotal)
        distances, indices = self.index.search(query_emb, actual_k)
        return [
            {"rank": rank, "image_path": self.image_paths[idx], "score": float(score)}
            for rank, (idx, score) in enumerate(zip(indices[0], distances[0]), start=1)
            if idx != -1
        ]

print("Configuración y funciones cargadas.")

---
## 2. Dataset: MS-COCO 2017 Val

Usamos el split **val2017** de MS-COCO: exactamente **5.000 imágenes** con **80 categorías** anotadas.

¿Por qué COCO?
- Categorías oficiales con IDs numéricos : ground truth limpio para Precision@K
- Una imagen puede tener múltiples categorías (un perro en la playa tiene `dog` + `person` + `outdoor`)
- Split exactamente documentado: reproducible por cualquier evaluador

La siguiente celda descarga las imágenes (~1 GB) y genera `metadata.csv`.

In [ ]:
COCO_IMAGES_URL      = "http://images.cocodataset.org/zips/val2017.zip"
COCO_ANNOTATIONS_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"

def _download_file(url, dest):
    print(f"Descargando {url} ...")
    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True) as bar:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
            bar.update(len(chunk))

config.DATA_DIR.mkdir(exist_ok=True)
config.EMBEDDINGS_DIR.mkdir(exist_ok=True)

# Imágenes
images_zip       = config.DATA_DIR / "val2017.zip"
extracted_images = config.DATA_DIR / "val2017"

if not config.IMAGES_DIR.exists() or not any(config.IMAGES_DIR.iterdir()):
    if not images_zip.exists():
        _download_file(COCO_IMAGES_URL, images_zip)
    print("Extrayendo imágenes ...")
    with zipfile.ZipFile(images_zip, "r") as zf:
        zf.extractall(config.DATA_DIR)
    if extracted_images.exists():
        if config.IMAGES_DIR.exists():
            config.IMAGES_DIR.rmdir()
        shutil.move(str(extracted_images), str(config.IMAGES_DIR))
    print(f"Imágenes listas : {config.IMAGES_DIR}")
else:
    print(f"Imágenes ya presentes : {config.IMAGES_DIR}")

# Anotaciones
annotations_zip = config.DATA_DIR / "annotations_trainval2017.zip"
if not config.ANNOTATIONS_PATH.exists():
    if not annotations_zip.exists():
        _download_file(COCO_ANNOTATIONS_URL, annotations_zip)
    print("Extrayendo anotaciones ...")
    with zipfile.ZipFile(annotations_zip, "r") as zf:
        zf.extractall(config.DATA_DIR)
    print(f"Anotaciones listas : {config.ANNOTATIONS_PATH}")
else:
    print(f"Anotaciones ya presentes : {config.ANNOTATIONS_PATH}")

# metadata.csv
if not config.METADATA_PATH.exists():
    print("Generando metadata.csv ...")
    with open(config.ANNOTATIONS_PATH, encoding="utf-8") as fh:
        coco_data = json.load(fh)
    cat_map       = {c["id"]: c["name"] for c in coco_data["categories"]}
    image_cats    = {}
    image_cat_ids = {}
    for ann in coco_data["annotations"]:
        iid, cid = ann["image_id"], ann["category_id"]
        image_cats.setdefault(iid, set()).add(cat_map.get(cid, "unknown"))
        image_cat_ids.setdefault(iid, set()).add(cid)
    rows = [
        {
            "image_id": img["id"],
            "path": str(config.IMAGES_DIR / img["file_name"]),
            "categories": "|".join(sorted(image_cats.get(img["id"], set()))),
            "category_ids": "|".join(str(c) for c in sorted(image_cat_ids.get(img["id"], set()))),
            "optional_description": "",
        }
        for img in coco_data["images"]
    ]
    pd.DataFrame(rows).to_csv(config.METADATA_PATH, index=False)
    print(f"metadata.csv generado - {len(rows)} imágenes")
else:
    print(f"metadata.csv ya presente : {config.METADATA_PATH}")

In [ ]:
# Cargar metadata
df = pd.read_csv(config.METADATA_PATH)
print(f"Total imágenes : {len(df)}")
print(f"Columnas       : {list(df.columns)}")
df.head(5)

In [ ]:
# Distribución de las 15 categorías más frecuentes
from collections import Counter

all_cats = []
for cats in df['categories'].dropna():
    all_cats.extend(cats.split('|'))

counter = Counter(all_cats)
top15 = counter.most_common(15)
labels, values = zip(*top15)

plt.figure(figsize=(12, 4))
plt.bar(labels, values, color='steelblue')
plt.title('Top 15 categorías COCO más frecuentes en val2017')
plt.xticks(rotation=45, ha='right')
plt.ylabel('# imágenes')
plt.tight_layout()
plt.show()

print(f"\nCategorías únicas en el dataset: {len(counter)}")

In [ ]:
# Mostrar 6 imágenes de muestra
sample = df.sample(6, random_state=42)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for ax, (_, row) in zip(axes.flatten(), sample.iterrows()):
    img = Image.open(row['path']).convert('RGB')
    ax.imshow(img)
    ax.set_title(row['categories'][:40], fontsize=8)
    ax.axis('off')

plt.suptitle('Muestra del dataset COCO val2017', y=1.02)
plt.tight_layout()
plt.show()

---
## 3. OpenCLIP y Embeddings Multimodales

### ¿Qué es un embedding?

Un **embedding** es una representación numérica densa de un objeto (texto, imagen, audio) en un espacio vectorial de alta dimensión.
Objetos semánticamente similares tienen embeddings cercanos - esto es la **hipótesis distribucional** aplicada a visión.

### ¿Por qué OpenCLIP?

CLIP (Contrastive Language-Image Pre-training) fue entrenado con **pares texto-imagen** usando **contrastive learning**:
- Dado un batch de N pares (imagen_i, texto_i), el modelo aprende a maximizar la similitud de los pares correctos
- y minimizar la similitud de los pares incorrectos
- Resultado: texto e imagen viven en el **mismo espacio vectorial** - se pueden comparar directamente

**OpenCLIP** es la reimplementación open-source. Usamos `ViT-B-32` entrenado en LAION-2B (2.3 mil millones de pares).

### L2-normalización (OBLIGATORIO)

Antes de indexar o comparar cualquier embedding, se **L2-normaliza**: cada vector se divide por su norma.
Resultado: vectores unitarios con $\|v\| = 1$.

Con vectores unitarios: **producto interno = similitud coseno**
$$\text{cos}(u, v) = \frac{u \cdot v}{\|u\| \|v\|} = u \cdot v \quad \text{(cuando } \|u\|=\|v\|=1\text{)}$$

In [ ]:
# Cargar modelo OpenCLIP
print(f"Cargando {config.MODEL_NAME} ({config.PRETRAINED}) en {config.DEVICE}...")
model, preprocess, tokenizer = load_model()

# Parámetros del modelo
total_params = sum(p.numel() for p in model.parameters())
print(f"Parámetros totales : {total_params:,}")
print(f"Dimensión embedding: {config.EMBEDDING_DIM}")

In [ ]:
# Demo: codificación de texto
sample_query = "dog running on the beach"
text_emb = encode_text(sample_query, model, tokenizer)

print(f"Query: '{sample_query}'")
print(f"Embedding shape : {text_emb.shape}")
print(f"Norma L2        : {np.linalg.norm(text_emb):.6f}  (debe ser ~1.0)")
print(f"Primeros 8 dims : {text_emb[0, :8]}")

In [ ]:
# Demo de similitud semántica entre consultas
queries = [
    "dog running on the beach",
    "puppy playing outside",
    "cat sleeping on a sofa",
    "airplane flying in the sky",
    "perro corriendo en la playa",  # español - prueba de robustez
]

embs = np.vstack([encode_text(q, model, tokenizer) for q in queries])
sim_matrix = embs @ embs.T  # producto interno = similitud coseno (vectores unitarios)

print("Matriz de similitud coseno entre consultas:")
print(f"{'':40}" + "".join(f"{i:8}" for i in range(len(queries))))
for i, q in enumerate(queries):
    row = "".join(f"{sim_matrix[i,j]:8.3f}" for j in range(len(queries)))
    print(f"{q[:38]:<40}{row}")

print("\n: Las consultas 0 y 1 (dog/puppy) deberían ser más similares entre sí que con cat o airplane.")
print(": La consulta 4 (español) debería ser similar a la consulta 0 (inglés) pero con menor puntuación.")

In [ ]:
if not config.IMAGE_EMBEDDINGS_PATH.exists():
    print("Generando embeddings (puede tardar varios minutos)...")
    img_paths_emb = df["path"].tolist()
    embeddings_emb = encode_images(img_paths_emb, model, preprocess)
    np.save(config.IMAGE_EMBEDDINGS_PATH, embeddings_emb)
    with open(config.IMAGE_PATHS_PATH, "wb") as fh:
        pickle.dump(img_paths_emb, fh)
    print(f"Embeddings guardados : {config.IMAGE_EMBEDDINGS_PATH}")
else:
    print(f"Embeddings ya presentes : {config.IMAGE_EMBEDDINGS_PATH}")

image_embeddings = np.load(config.IMAGE_EMBEDDINGS_PATH)
with open(config.IMAGE_PATHS_PATH, "rb") as fh:
    image_paths = pickle.load(fh)

print(f"Shape embeddings : {image_embeddings.shape}  (N_imágenes x dim)")
print(f"Norma media      : {np.linalg.norm(image_embeddings, axis=1).mean():.6f}  (debe ser ~1.0)")
print(f"Total paths      : {len(image_paths)}")

---
## 4. Indexación con FAISS

### ¿Por qué FAISS?

Con 5.000 imágenes, podríamos calcular la similitud coseno directamente con NumPy.
Pero FAISS es el estándar industrial para búsqueda vectorial eficiente - y su uso demuestra
comprensión de cómo se escala la búsqueda semántica en sistemas reales.

### ¿Por qué `IndexFlatIP`?

- **Flat** = almacena todos los vectores sin compresión : búsqueda **exacta** (no aproximada)
- **IP** = Inner Product - el tipo de distancia que usa
- Con vectores L2-normalizados: IP = cosine similarity : resultados idénticos a cosine, pero via FAISS
- Para 5.000 vectores de dim 512: el índice ocupa ~10 MB en RAM - trivial

**Alternativas** (descartadas para este proyecto):
- `IndexIVFFlat`: aproximado, requiere fase de entrenamiento. Overhead injustificado a esta escala.
- `IndexHNSWFlat`: para millones de vectores. Añade parámetros y complejidad innecesaria.

> **Regla**: usar el índice más simple que satisfaga los requisitos de performance. A 5K vectores, la búsqueda exacta tarda microsegundos.

In [ ]:
if not config.FAISS_INDEX_PATH.exists():
    print("Construyendo índice FAISS...")
    _idx = build_index(image_embeddings)
    save_index(_idx, config.FAISS_INDEX_PATH)
    print(f"Índice guardado : {config.FAISS_INDEX_PATH}")
else:
    print(f"Índice ya presente : {config.FAISS_INDEX_PATH}")

index = load_index(config.FAISS_INDEX_PATH)
print(f"Tipo de índice    : {type(index).__name__}")
print(f"Vectores en índice: {index.ntotal}")
print(f"Dimensión         : {index.d}")

In [ ]:
# Verificación: buscar la imagen más similar a sí misma
# Un embedding debe ser su propio vecino más cercano con puntuación = 1.0
test_emb = image_embeddings[0:1]  # primera imagen
distances, indices = index.search(test_emb, k=3)

print("Prueba de auto-recuperación (la imagen debe ser su propio top-1):")
for rank, (idx, dist) in enumerate(zip(indices[0], distances[0]), 1):
    print(f"  Rank {rank}: idx={idx}  sim_coseno={dist:.6f}  path={Path(image_paths[idx]).name}")

assert indices[0][0] == 0 and abs(distances[0][0] - 1.0) < 1e-4, "La prueba de auto-recuperación falló!"
print("\nOK Auto-recuperación correcta: puntuación=1.0 y es el mismo vector.")

---
## 5. Demo de Búsqueda Semántica

Ahora combinamos todo: dado un texto, generamos su embedding, lo comparamos contra
el índice FAISS, y mostramos las imágenes más similares.

**Flujo de una búsqueda:**

1. `query (str)`
2. Tokenizer + OpenCLIP text encoder : `text_embedding (1x512, float32)`
3. L2-normalize : `unit_vector (norma = 1.0)`
4. FAISS `index.search(k=5)` : `top-5 (indices, cosine_scores)`
5. Mapeo de índices a `image_paths` : resultados visuales

In [ ]:
# Inicializar el motor de búsqueda
engine = SearchEngine()
print("SearchEngine listo.")

In [ ]:
def show_results(query: str, k: int = 5):
    results = engine.search(query, k=k)
    fig, axes = plt.subplots(1, k, figsize=(4*k, 4))
    if k == 1:
        axes = [axes]
    for ax, r in zip(axes, results):
        img = Image.open(r['image_path']).convert('RGB')
        ax.imshow(img)
        ax.set_title(f"#{r['rank']}  {r['score']:.3f}", fontsize=10)
        ax.axis('off')
    for ax in axes[len(results):]:
        ax.axis('off')
    plt.suptitle(f'Query: "{query}"', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Caso exitoso - categoría bien representada
show_results("dog running on the beach")

In [ ]:
show_results("giraffe eating leaves from a tree")

In [ ]:
show_results("pizza on a wooden table")

In [ ]:
show_results("person surfing on ocean waves")

In [ ]:
# Caso con resultados ambiguos - para análisis de limitaciones
show_results("happy emotion feeling")
print(": Concepto abstracto sin representación visual directa en el dataset.")

In [ ]:
# Prueba español vs inglés - robustez del modelo
print("=== Inglés ===")
r_en = engine.search("dog running on the beach", k=5)
print(f"Puntuaciones: {[round(r['score'],3) for r in r_en]}")

print("\n=== Español ===")
r_es = engine.search("perro corriendo en la playa", k=5)
print(f"Puntuaciones: {[round(r['score'],3) for r in r_es]}")

print("\n: Las puntuaciones en español suelen ser menores (~10-20%) porque el modelo")
print("  fue entrenado predominantemente en texto en inglés.")

---
## 6. Evaluación Cuantitativa

### Definición de relevancia

Una imagen recuperada es **relevante** si comparte al menos una categoría COCO con la
categoría objetivo de la consulta.

Ejemplo: query `"dog running on the beach"` : `target_category_id = 18 (dog)`
: relevante = cualquier imagen anotada con `dog` en COCO.

### Métricas

$$P@K = \frac{|\text{relevantes en top-}K|}{K}$$

$$R@K = \frac{|\text{relevantes en top-}K|}{|\text{total relevantes en dataset}|}$$

### Criterio de éxito

$$\overline{P@5} \geq 0.70$$

In [ ]:
# Cargar anotaciones COCO para ground truth
annotations = parse_coco_annotations(config.ANNOTATIONS_PATH)
category_index = build_image_category_index(annotations)

print(f"Imágenes en el índice de categorías: {len(category_index)}")
# Mostrar ejemplo: categorías de la primera imagen
first_id = list(category_index.keys())[0]
cat_map = {cat['id']: cat['name'] for cat in annotations['categories']}
cats = [cat_map[c] for c in category_index[first_id]]
print(f"Ejemplo image_id={first_id}: categorías = {cats}")

In [ ]:
EVAL_QUERIES = [
    {"query_text": "dog running on the beach",             "target_category": "dog",           "target_category_id": 18},
    {"query_text": "cat sleeping on a couch",              "target_category": "cat",            "target_category_id": 17},
    {"query_text": "person riding a bicycle",              "target_category": "bicycle",        "target_category_id": 2},
    {"query_text": "car driving on a city street",         "target_category": "car",            "target_category_id": 3},
    {"query_text": "airplane flying in the sky",           "target_category": "airplane",       "target_category_id": 5},
    {"query_text": "elephant in the wild",                 "target_category": "elephant",       "target_category_id": 22},
    {"query_text": "zebra standing on dry grass",          "target_category": "zebra",          "target_category_id": 24},
    {"query_text": "giraffe eating leaves from a tree",    "target_category": "giraffe",        "target_category_id": 25},
    {"query_text": "pizza on a wooden table",              "target_category": "pizza",          "target_category_id": 59},
    {"query_text": "people playing sports outdoors",       "target_category": "person",         "target_category_id": 1},
    {"query_text": "horse galloping in a green field",     "target_category": "horse",          "target_category_id": 19},
    {"query_text": "bird perched on a branch",             "target_category": "bird",           "target_category_id": 16},
    {"query_text": "bus stopped at a city street",         "target_category": "bus",            "target_category_id": 6},
    {"query_text": "boat sailing on the water",            "target_category": "boat",           "target_category_id": 9},
    {"query_text": "person working on a laptop",           "target_category": "laptop",         "target_category_id": 73},
    {"query_text": "food served on a dining table",        "target_category": "dining table",   "target_category_id": 67},
    {"query_text": "glass bottle on a shelf",              "target_category": "bottle",         "target_category_id": 44},
    {"query_text": "chair next to a table indoors",        "target_category": "chair",          "target_category_id": 62},
    {"query_text": "truck driving on a highway",           "target_category": "truck",          "target_category_id": 8},
    {"query_text": "person surfing on ocean waves",        "target_category": "surfboard",      "target_category_id": 42},
    {"query_text": "cow grazing in a green field",         "target_category": "cow",            "target_category_id": 21},
    {"query_text": "baseball player holding a bat",        "target_category": "baseball bat",   "target_category_id": 39},
    {"query_text": "person holding an umbrella in rain",   "target_category": "umbrella",       "target_category_id": 28},
    {"query_text": "skier going down a snowy mountain",    "target_category": "skis",           "target_category_id": 35},
    {"query_text": "sheep on a hillside meadow",           "target_category": "sheep",          "target_category_id": 20},
]
K_VALUES = [1, 3, 5]

print(f"Total queries de evaluación: {len(EVAL_QUERIES)}")
report = evaluate(engine, EVAL_QUERIES, category_index, k_values=K_VALUES)
print("Evaluación completa.")

In [ ]:
# Tabla de resultados
rows = []
for r in report['query_results']:
    rows.append({
        'query': r['query_text'],
        'target': r['target_category'],
        'P@1': round(r['precision_at_1'], 2),
        'P@3': round(r['precision_at_3'], 2),
        'P@5': round(r['precision_at_5'], 2),
        'R@5': round(r['recall_at_5'], 4),
    })
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))
print(f"\nAvg P@1: {report['avg_precision_at_1']:.4f}")
print(f"Avg P@3: {report['avg_precision_at_3']:.4f}")
print(f"Avg P@5: {report['avg_precision_at_5']:.4f}  (criterio de exito >= 0.70)")
print(f"\nAvg R@1: {report['avg_recall_at_1']:.4f}")
print(f"Avg R@3: {report['avg_recall_at_3']:.4f}")
print(f"Avg R@5: {report['avg_recall_at_5']:.4f}")
print(f"\nResultado: {'OK ÉXITO' if report['success'] else '[X] BAJO UMBRAL'}")

In [ ]:
# Visualización de Precision@K por query
queries_short = [r['query_text'][:35] for r in report['query_results']]
p1 = [r['precision_at_1'] for r in report['query_results']]
p3 = [r['precision_at_3'] for r in report['query_results']]
p5 = [r['precision_at_5'] for r in report['query_results']]

x = range(len(queries_short))
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(x, p1, 'o-', label='P@1', alpha=0.8)
ax.plot(x, p3, 's-', label='P@3', alpha=0.8)
ax.plot(x, p5, '^-', label='P@5', alpha=0.8)
ax.axhline(0.70, color='red', linestyle='--', label='Umbral 0.70')
ax.set_xticks(list(x))
ax.set_xticklabels(queries_short, rotation=45, ha='right', fontsize=7)
ax.set_ylim(-0.05, 1.05)
ax.set_ylabel('Precision')
ax.set_title('Precision@K por query')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Análisis de errores - consultas con P@5 < 0.70
failures = [r for r in report['query_results'] if r['precision_at_5'] < 0.70]
print(f"Consultas por debajo del umbral (P@5 < 0.70): {len(failures)}")
print()
for r in failures:
    print(f"  Consulta : '{r['query_text']}'")
    print(f"  Categoría objetivo: {r['target_category']} (id={r['target_category_id']})")
    print(f"  P@5      : {r['precision_at_5']:.2f}")
    print(f"  Relevantes en dataset: {r['total_relevant_in_dataset']}")
    print()

if failures:
    print("Posibles causas de fallo:")
    print("  - Categoría poco representada en val2017 (pocas imágenes relevantes)")
    print("  - Consulta demasiado específica (atributo visual que el modelo no captura bien)")
    print("  - Similitud visual fuerte con otras categorías (confusión semántica)")
    print("  - Ambigüedad: la consulta puede interpretarse de múltiples formas")

---
## 7. Fundamento Matemático y Teórico

### 7.1 Embeddings y Similitud Vectorial

Un **embedding** $\phi: X \to \mathbb{R}^d$ mapea un objeto (imagen o texto) a un vector de dimensión $d$.

La **similitud coseno** mide el ángulo entre dos vectores, independientemente de su magnitud:

$$\text{sim}(u, v) = \cos(\theta) = \frac{u \cdot v}{\|u\| \cdot \|v\|}$$

Con vectores L2-normalizados ($\|u\| = \|v\| = 1$), esto simplifica a:

$$\text{sim}(u, v) = u \cdot v \quad \in [-1, 1]$$

: FAISS `IndexFlatIP` computa exactamente esto en cada búsqueda.

### 7.2 Contrastive Learning y OpenCLIP

CLIP es entrenado con la **pérdida InfoNCE** (Noise-Contrastive Estimation).

Dado un batch de $N$ pares $(I_i, T_i)$ (imagen $i$, texto $i$), el modelo maximiza:

$$\mathcal{L}_{\text{InfoNCE}} = -\frac{1}{N} \sum_{i=1}^{N} \log \frac{\exp(\text{sim}(\phi_I(I_i),\, \phi_T(T_i)) / \tau)}{\sum_{j=1}^{N} \exp(\text{sim}(\phi_I(I_i),\, \phi_T(T_j)) / \tau)}$$

Donde:
- $\phi_I$ y $\phi_T$ son los encoders de imagen y texto
- $\tau$ es la temperatura (learnable)
- El numerador: similitud del par correcto $(I_i, T_i)$
- El denominador: similitud del par correcto + todos los pares incorrectos $(I_i, T_j), j \neq i$

**Efecto**: el modelo aprende un espacio donde `embed("dog")` ~ `embed(foto_de_perro)`.
Esto habilita la búsqueda zero-shot texto-imagen sin entrenamiento adicional.

### 7.3 Búsqueda de Vecinos Más Cercanos con FAISS

El problema Top-K NN: dado un query $q$ y un conjunto $\mathcal{D} = \{v_1, ..., v_N\}$, encontrar:

$$\text{top-}K = \underset{v \in \mathcal{D}}{\text{argmax}_K} \; \text{sim}(q, v)$$

`IndexFlatIP` resuelve esto con complejidad $O(N \cdot d)$ - búsqueda exacta.
Para $N = 5000$, $d = 512$: ~2.5M operaciones por query : microsegundos en CPU.

In [ ]:
# Demo matemático: L2-normalization y similitud coseno
v1 = np.array([3.0, 4.0])  # norma = 5
v2 = np.array([1.5, 2.0])  # norma = 2.5, misma dirección que v1

# Sin normalizar
ip_raw = np.dot(v1, v2)
cos_raw = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
print(f"Sin normalizar  : producto interno={ip_raw:.2f}, coseno={cos_raw:.4f}")

# Con normalización L2
v1_n = v1 / np.linalg.norm(v1)
v2_n = v2 / np.linalg.norm(v2)
ip_norm = np.dot(v1_n, v2_n)
print(f"L2-normalizados : producto interno={ip_norm:.4f} == coseno={cos_raw:.4f}  OK")
print("\n: Con vectores unitarios, producto interno y similitud coseno son idénticos.")
print("  Por eso FAISS IndexFlatIP + L2-normalize = búsqueda por similitud coseno exacta.")

In [ ]:
# Demo de InfoNCE (versión simplificada)
import torch.nn.functional as F

# Simular un mini-batch de 4 pares (imagen, texto)
torch.manual_seed(42)
image_feats = F.normalize(torch.randn(4, 512), dim=-1)
text_feats  = F.normalize(torch.randn(4, 512), dim=-1)

temperature = 0.07
logits = (image_feats @ text_feats.T) / temperature  # matriz 4x4
labels = torch.arange(4)  # etiquetas: par (i, i)

loss = F.cross_entropy(logits, labels)
print("Demo de la pérdida InfoNCE:")
print(f"  Forma de logits : {logits.shape}  (NxN - cada fila: imagen vs todos los textos)")
print(f"  Etiquetas reales: {labels.tolist()}  (diagonal = pares correctos)")
print(f"  Pérdida InfoNCE : {loss.item():.4f}")
print()
print(": OpenCLIP minimiza esta pérdida sobre 2.3B pares imagen-texto (LAION-2B).")
print("  Resultado: imagen y texto del mismo concepto tienen embeddings cercanos.")

---
## 8. Conclusiones

### Lo que demostró este proyecto

1. **Los embeddings multimodales funcionan**: OpenCLIP ViT-B-32 produce representaciones que conectan texto e imagen en el mismo espacio vectorial sin fine-tuning adicional.

2. **La búsqueda vectorial es viable**: FAISS `IndexFlatIP` sobre 5.000 vectores de 512 dimensiones responde en microsegundos con resultados exactos.

3. **La evaluación cuantitativa es esencial**: Precision@K y Recall@K objetivan el rendimiento del sistema más allá de capturas de pantalla seleccionadas.

### Limitaciones identificadas

| Limitación | Causa | Mitigación posible |
|---|---|---|
| Peor rendimiento en español | Modelo entrenado en inglés | Modelo multilingüe (NLLB-CLIP, M-CLIP) |
| Consultas abstractas fallan | CLIP no captura emociones/conceptos no visuales | Fine-tuning en dominio específico |
| Categorías poco representadas | COCO val2017 desbalanceado | Dataset más balanceado |
| Precisión limitada por categoría amplia | "person" es demasiado amplia | Usar sub-categorías o atributos |

### Pipeline completo integrado

Este proyecto integra en un prototipo funcional:
**NLP** (tokenización de texto) - **Transformers** (ViT + text encoder) - **Embeddings** (espacio compartido) - **Modelos multimodales** (OpenCLIP) - **Búsqueda vectorial** (FAISS) - **Evaluación** (Precision@K, Recall@K)